In [ ]:
""" Imports """
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

from simulation_encoder.loaders.loader_retrieval import load_loaders
from simulation_encoder.models.model_retrieval import load_models


In [ ]:
study_name = "cae-latentdim-gastruloid"
image_dir = "data"
num_timepoints = 9

results_dir = f"results/{study_name}"

all_models = load_models(results_dir, num_timepoints=num_timepoints, best_models_flag=False)
all_loaders = load_loaders(results_dir, image_dir)

print(f"Models: {list(all_models.keys())}")
print(f"Datasets: {list(all_loaders.keys())}")

all_samples = {}
all_labels = {}
data_loaders = {}
random_indices = {}

for dataset_name, loader in all_loaders.items():
    data_loaders[dataset_name] = {
        "train": loader.get_dataloader('train'),
        "test": loader.get_dataloader('test'),
        "full": loader,
    }
    
    loader_test_len = len(data_loaders[dataset_name]["test"].dataset)
    
    if dataset_name not in random_indices:
        random_indices[dataset_name] = np.random.choice(loader_test_len, 5, replace=False)
    
    indices = random_indices[dataset_name]
    all_samples[dataset_name] = []
    all_labels[dataset_name] = []
    
    for i in indices:
        image, label = data_loaders[dataset_name]["test"].dataset[i]
        sample_id = loader._get_data_feature(i, "sample_id")
        array_num = int(sample_id.split('_')[0][-1])
        all_samples[dataset_name].append(image)
        all_labels[dataset_name].append("Euploid" if array_num == 1 else "Anueploid")